# ALY 6130 — Risk Management | Assignment 3
## Programmatic Risk Heat Map
### Amazon AI-Powered Last-Mile Delivery Optimization Platform Acquisition

**Group 6:** Sandra Amponsah | Truc Linh Hoang | Victoria Kpetigo  
**Northeastern University | June 2026**

---

This notebook reproduces the risk heat map from the accompanying Excel file **programmatically in Python**, as required by the Assignment 3 rubric.


**Axes (from Risk Calculation Sheet — NOT 1 to 10):**
- **X-axis = Likelihood:** `{1, 3, 5, 7, 9}`
- **Y-axis = Impact:** `{1, 2, 4, 6, 8, 9}`
- **Risk Score = Likelihood × Impact**

**Thresholds (from Risk Calculation Sheet):**
| Severity | Score Range | Color |
|---|---|---|
| Very High | ≥ 54 | Red |
| High | 40 – 53 | Orange |
| Moderate | 18 – 39 | Yellow |
| Low | < 18 | Green |
| Positive Risk ★ | — | Green (lighter) |

In [1]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import numpy as np

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'figure.dpi': 130,
})
print('Setup complete.')

ModuleNotFoundError: No module named 'matplotlib'

---
## Step 1 — Define the Grid (from Risk Calculation Sheet)

The grid below is taken **directly from the Heatmap sheet** of the Excel file.  
Each cell contains the score and the risk IDs that fall at that Likelihood × Impact coordinate.

In [ ]:
# ── Axis values — taken directly from the Risk Calculation Sheet
# X-axis: Likelihood (columns in the Excel heatmap)
likelihood_values  = [1,   3,   5,   7,   9]
likelihood_labels  = [
    'P = 1\nVery Unlikely',
    'P = 3\nSomewhat Unlikely',
    'P = 5\n50-50',
    'P = 7\nSomewhat Likely',
    'P = 9\nVery Likely',
]

# Y-axis: Impact (rows in the Excel heatmap, top = highest)
impact_values  = [9,   8,   6,   4,   2,   1]
impact_labels  = [
    'Extremely High (9)',
    'High (8)',
    'Somewhat High (6)',
    'Moderate (4)',
    'Somewhat Low (2)',
    'Very Low (1)',
]

# ── Cell contents — copied exactly from the Excel Heatmap sheet
# Format: { (likelihood, impact): (score, 'risk labels', is_positive_opportunity) }
# Row order matches Excel: Impact 9 → 8 → 6 → 4 → 2 → 1 (top to bottom)

heatmap_cells = {
    # Impact = 9 (Extremely High)
    (1, 9): (9,  '',                        False),
    (3, 9): (27, 'R25, R54',                False),
    (5, 9): (45, 'R34, R46',                False),
    (7, 9): (63, 'R16',                     False),
    (9, 9): (81, 'R6, R43',                 False),

    # Impact = 8 (High)
    (1, 8): (8,  'R53',                     False),
    (3, 8): (24, 'R33, R37, R56, R60',      False),
    (5, 8): (40, 'R15, R27, R42, R55',      False),
    (7, 8): (56, 'R7, R44',                 False),
    (9, 8): (72, 'R5, R24',                 True),   # ★ Positive opportunity

    # Impact = 6 (Somewhat High)
    (1, 6): (6,  '',                        False),
    (3, 6): (18, 'R14, R23, R48, R57, R59', False),
    (5, 6): (30, 'R26, R28, R41, R45, R58', False),
    (7, 6): (42, 'R8, R10, R18, R20,\nR21, R35, R49, R52', False),
    (9, 6): (54, 'R4',                      False),

    # Impact = 4 (Moderate)
    (1, 4): (4,  'R22',                     False),
    (3, 4): (12, 'R9, R29',                 False),
    (5, 4): (20, 'R17, R19, R32, R36, R47', False),
    (7, 4): (28, 'R3, R40',                 False),
    (9, 4): (36, 'R13, R51',                False),

    # Impact = 2 (Somewhat Low)
    (1, 2): (2,  '',                        False),
    (3, 2): (6,  '',                        False),
    (5, 2): (10, 'R2, R12',                 False),
    (7, 2): (14, 'R31',                     False),
    (9, 2): (18, 'R39',                     False),

    # Impact = 1 (Very Low)
    (1, 1): (1,  'R38',                     False),
    (3, 1): (3,  '',                        False),
    (5, 1): (5,  'R50',                     False),
    (7, 1): (7,  'R11',                     False),
    (9, 1): (9,  'R1, R30',                 False),
}

print('Grid loaded: 5 Likelihood columns × 6 Impact rows = 30 cells')
print(f'Likelihood axis (X): {likelihood_values}')
print(f'Impact axis (Y):     {impact_values}')

---
## Step 2 — Define Thresholds and Colors

Thresholds are taken directly from the Risk Calculation Sheet legend:  
`GREEN (Low): P×I < 18 | YELLOW (Moderate): 18–39 | ORANGE (High): 40–53 | RED (Very High): ≥ 54`

In [ ]:
# ── Colors matching the Excel heatmap
COLOR_VERY_HIGH = '#FF9999'   # Red   — score ≥ 54
COLOR_HIGH      = '#FFCCCC'   # Orange/Pink — score 40–53
COLOR_MODERATE  = '#FFF2CC'   # Yellow — score 18–39
COLOR_LOW       = '#E2EFDA'   # Green  — score < 18
COLOR_POSITIVE  = '#C6EFCE'   # Bright green — positive risk / opportunity

TEXT_VERY_HIGH  = '#8B0000'
TEXT_HIGH       = '#C00000'
TEXT_MODERATE   = '#7D6608'
TEXT_LOW        = '#375623'
TEXT_POSITIVE   = '#1E5631'

def get_colors(score, is_positive=False):
    """Return (face_color, text_color) based on score and risk type."""
    if is_positive:
        return COLOR_POSITIVE, TEXT_POSITIVE
    if score >= 54: return COLOR_VERY_HIGH, TEXT_VERY_HIGH
    if score >= 40: return COLOR_HIGH,      TEXT_HIGH
    if score >= 18: return COLOR_MODERATE,  TEXT_MODERATE
    return COLOR_LOW, TEXT_LOW

# Print threshold table
print('SEVERITY THRESHOLDS (from Risk Calculation Sheet)')
print('=' * 65)
thresholds = [
    ('Very High', 'Score ≥ 54',  'Red',    'Board-level escalation; immediate treatment plan'),
    ('High',      'Score 40–53', 'Orange', 'Executive action; active mitigation; monthly KRI review'),
    ('Moderate',  'Score 18–39', 'Yellow', 'Monitor; assign owner; quarterly review'),
    ('Low',       'Score < 18',  'Green',  'Accept with documented residual risk; annual review'),
    ('★ Positive (R5, R24)','Opportunity', 'Bright Green', 'Exploit; track as strategic KPI; accelerate capture'),
]
for sev, rng, color, action in thresholds:
    print(f'  {sev:<12} {rng:<14} {color:<14} {action}')

---
## Step 3 — Draw the Heat Map

The chart below mirrors the Excel Heatmap sheet exactly:  
- **X-axis = Likelihood** (columns: P=1, 3, 5, 7, 9)  
- **Y-axis = Impact** (rows: I=9 at top → I=1 at bottom)  
- Each cell shows its **Score** and the **risk IDs** from the register

In [ ]:
n_cols = len(likelihood_values)   # 5
n_rows = len(impact_values)       # 6

fig, ax = plt.subplots(figsize=(16, 11))
fig.patch.set_facecolor('white')
ax.set_xlim(0, n_cols)
ax.set_ylim(0, n_rows)
ax.set_aspect('equal')

# ── Draw every cell
for col_i, l_val in enumerate(likelihood_values):       # X: left → right
    for row_i, i_val in enumerate(impact_values):       # Y: top → bottom in data
        # In matplotlib: row 0 = bottom, so invert
        y_pos = n_rows - 1 - row_i
        x_pos = col_i

        score, risk_ids, is_pos = heatmap_cells.get((l_val, i_val), (l_val*i_val, '', False))
        fc, tc = get_colors(score, is_pos)

        # Cell rectangle
        rect = plt.Rectangle(
            (x_pos, y_pos), 1, 1,
            facecolor=fc, edgecolor='#999999', linewidth=1.0, zorder=1
        )
        ax.add_patch(rect)

        # Score — top-left of cell
        ax.text(
            x_pos + 0.07, y_pos + 0.93,
            f'Score = {score}',
            ha='left', va='top', fontsize=8, color=tc,
            fontweight='bold', zorder=3
        )

        # Risk IDs — centred in cell
        if risk_ids:
            label = risk_ids
            if is_pos:
                label += '  ★'
            ax.text(
                x_pos + 0.5, y_pos + 0.44,
                label,
                ha='center', va='center', fontsize=7.8,
                color=tc, fontweight='bold', zorder=4,
                bbox=dict(boxstyle='round,pad=0.25', facecolor='white',
                          alpha=0.75, edgecolor='none')
            )

# ── X-axis ticks and labels (Likelihood)
ax.set_xticks([i + 0.5 for i in range(n_cols)])
ax.set_xticklabels(likelihood_labels, fontsize=9.5, color='#1F4E79')
ax.tick_params(axis='x', length=0, pad=6)

# ── Y-axis ticks and labels (Impact — highest at top)
ax.set_yticks([i + 0.5 for i in range(n_rows)])
# Impact labels: bottom cell = impact_values[-1], top = impact_values[0]
ax.set_yticklabels(impact_labels[::-1], fontsize=9.5, color='#1F4E79')
ax.tick_params(axis='y', length=0, pad=6)

# ── Axis labels
ax.set_xlabel(
    'LIKELIHOOD  (X-axis)   —   Scale: {1, 3, 5, 7, 9}  (from Risk Calculation Sheet)',
    fontsize=12, fontweight='bold', labelpad=14, color='#1F4E79'
)
ax.set_ylabel(
    'IMPACT  (Y-axis)   —   Scale: {1, 2, 4, 6, 8, 9}  (from Risk Calculation Sheet)',
    fontsize=12, fontweight='bold', labelpad=14, color='#1F4E79'
)

ax.set_title(
    'Risk Heat Map — Amazon AI-Powered Last-Mile Delivery Optimization Platform Acquisition\n'
    'ALY 6130  |  Group 6  |  60 Risks  |  Programmatic (Python)  |  June 2026',
    fontsize=13, fontweight='bold', pad=16, color='#1F4E79'
)

# ── Legend (matches Excel legend exactly)
legend_handles = [
    mpatches.Patch(facecolor=COLOR_VERY_HIGH, edgecolor='#999999',
                   label='Very High Risk  (Score ≥ 54)   — Board-level escalation'),
    mpatches.Patch(facecolor=COLOR_HIGH,      edgecolor='#999999',
                   label='High Risk  (Score 40–53)   — Executive action; monthly KRI review'),
    mpatches.Patch(facecolor=COLOR_MODERATE,  edgecolor='#999999',
                   label='Moderate Risk  (Score 18–39)   — Monitor; quarterly review'),
    mpatches.Patch(facecolor=COLOR_LOW,       edgecolor='#999999',
                   label='Low Risk  (Score < 18)   — Accept; annual review'),
    mpatches.Patch(facecolor=COLOR_POSITIVE,  edgecolor='#375623',
                   label='★  Positive Risk / Opportunity   — Exploit; track as strategic KPI'),
]
ax.legend(
    handles=legend_handles,
    loc='upper left',
    bbox_to_anchor=(0.0, -0.13),
    ncol=2,
    fontsize=9,
    framealpha=0.96,
    edgecolor='#AAAAAA',
    title='LEGEND  (from Risk Calculation Sheet)',
    title_fontsize=9.5,
)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.spines['left'].set_visible(False)

plt.tight_layout(rect=[0, 0.10, 1, 1])
plt.savefig('A3_heatmap_python.png', bbox_inches='tight', dpi=150)
plt.show()
print('Heat map saved: A3_heatmap_python.png')

---
## Step 4 — Plot Risk Scores in Relation to Total Risk Score

All 60 risk scores are plotted as a bar chart showing where each sits relative to the severity thresholds. R5 and R24 are positive opportunities and are shown in green.

In [ ]:
import pandas as pd

# Full risk register — ID, Likelihood, Impact (from Risk Register sheet)
register = [
    (1,9,1),(2,5,2),(3,7,4),(4,9,6),(5,9,8),(6,9,9),(7,7,8),(8,7,6),
    (9,3,4),(10,7,6),(11,7,1),(12,5,2),(13,9,4),(14,3,6),(15,5,8),
    (16,7,9),(17,5,4),(18,7,6),(19,5,4),(20,7,6),(21,7,6),(22,1,4),
    (23,3,6),(24,9,8),(25,3,9),(26,5,6),(27,5,8),(28,5,6),(29,3,4),
    (30,9,1),(31,7,2),(32,5,4),(33,3,8),(34,5,9),(35,7,6),(36,5,4),
    (37,3,8),(38,1,1),(39,9,2),(40,7,4),(41,5,6),(42,5,8),(43,9,9),
    (44,7,8),(45,5,6),(46,5,9),(47,5,4),(48,3,6),(49,7,6),(50,5,1),
    (51,9,4),(52,7,6),(53,1,8),(54,3,9),(55,5,8),(56,3,8),(57,3,6),
    (58,5,6),(59,3,6),(60,3,8),
]
# R5 and R24 are positive opportunities (consistent with heatmap)
positive_ids = {5, 24}

df = pd.DataFrame(register, columns=['ID','Likelihood','Impact'])
df['Score'] = df['Likelihood'] * df['Impact']
df['IsPositive'] = df['ID'].isin(positive_ids)
df['Severity'] = df['Score'].apply(
    lambda s: 'Very High' if s>=54 else 'High' if s>=40 else 'Moderate' if s>=18 else 'Low'
)
df_sorted = df.sort_values('Score', ascending=False).reset_index(drop=True)

# Bar colors
def bar_color(row):
    if row['IsPositive']: return COLOR_POSITIVE
    s = row['Score']
    if s >= 54: return COLOR_VERY_HIGH
    if s >= 40: return COLOR_HIGH
    if s >= 18: return COLOR_MODERATE
    return COLOR_LOW

def edge_color(row):
    if row['IsPositive']: return TEXT_POSITIVE
    s = row['Score']
    if s >= 54: return TEXT_VERY_HIGH
    if s >= 40: return TEXT_HIGH
    if s >= 18: return TEXT_MODERATE
    return TEXT_LOW

colors = [bar_color(r) for _, r in df_sorted.iterrows()]
edges  = [edge_color(r) for _, r in df_sorted.iterrows()]
labels = [f"R{int(r['ID'])}" for _, r in df_sorted.iterrows()]

fig, ax = plt.subplots(figsize=(18, 8))
fig.patch.set_facecolor('white')

bars = ax.bar(
    range(len(df_sorted)), df_sorted['Score'],
    color=colors, edgecolor=edges, linewidth=0.9, width=0.75
)

# Threshold lines
ax.axhline(54, color=TEXT_VERY_HIGH, linestyle='--', linewidth=1.8, alpha=0.85,
           label='Very High threshold  (Score = 54)')
ax.axhline(40, color=TEXT_HIGH,      linestyle='--', linewidth=1.5, alpha=0.75,
           label='High threshold  (Score = 40)')
ax.axhline(18, color=TEXT_MODERATE,  linestyle='--', linewidth=1.3, alpha=0.65,
           label='Moderate threshold  (Score = 18)')

# Threshold labels on right
for val, label, color in [
    (54, 'Very High (≥54)',  TEXT_VERY_HIGH),
    (40, 'High (40–53)',     TEXT_HIGH),
    (18, 'Moderate (18–39)', TEXT_MODERATE),
]:
    ax.text(len(df_sorted)-0.3, val+1.2, label,
            ha='right', va='bottom', fontsize=8.5,
            color=color, fontweight='bold')

# Zone shading
ax.axhspan(54, 85, alpha=0.05, color='red',    zorder=0)
ax.axhspan(40, 54, alpha=0.05, color='orange', zorder=0)
ax.axhspan(18, 40, alpha=0.05, color='yellow', zorder=0)
ax.axhspan(0,  18, alpha=0.05, color='green',  zorder=0)

ax.set_xticks(range(len(df_sorted)))
ax.set_xticklabels(labels, fontsize=7, rotation=45, ha='right')
ax.set_ylabel('Risk Score  (Likelihood × Impact)', fontsize=11, fontweight='bold', color='#1F4E79')
ax.set_xlabel('Risk ID', fontsize=11, fontweight='bold', labelpad=8, color='#1F4E79')
ax.set_title(
    'All 60 Risk Scores Plotted Against Severity Thresholds\n'
    'Amazon AI Last-Mile Acquisition  |  ALY 6130 Group 6  |  June 2026  |  R5 & R24 = Positive Opportunities',
    fontsize=13, fontweight='bold', color='#1F4E79', pad=12
)
ax.set_ylim(0, 88)
ax.legend(fontsize=9, framealpha=0.95, edgecolor='#AAAAAA')
ax.grid(axis='y', alpha=0.25, linestyle=':')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('A3_risk_scores_vs_thresholds.png', bbox_inches='tight', dpi=150)
plt.show()

# Count by severity
print('Risks by severity threshold:')
df_neg = df[~df['IsPositive']]  # exclude positive opportunities from severity count
for sev, rng in [('Very High','≥54'),('High','40–53'),('Moderate','18–39'),('Low','<18')]:
    n = len(df_neg[df_neg['Severity']==sev])
    print(f'  {sev:<12} ({rng}): {n} risks')

---
## Rubric Compliance

| Requirement | Status |
|---|---|
| Heat map using **Python programmatically** | ✅ Step 3 |
| **X-axis = Likelihood** | ✅ Values `{1, 3, 5, 7, 9}` from Risk Calculation Sheet |
| **Y-axis = Impact** | ✅ Values `{1, 2, 4, 6, 8, 9}` from Risk Calculation Sheet |
| **Do NOT use 1 to 10** | ✅ Exact scale from sheet used — not 1–10 |
| **Reflects what is in the Risk Calculation Sheet** | ✅ Every cell, score, and risk ID matches the Excel Heatmap sheet |
| **Plot scores in relation to total risk score** | ✅ Step 4 — bar chart of all 60 scores vs. thresholds |
| **Document High, Medium, Low thresholds** | ✅ Step 2 + threshold lines on both charts |
| **Red = High, Yellow = Medium, Green = Low** | ✅ Very High=Red, High=Orange-Red, Moderate=Yellow, Low=Green |